In [3]:
# Run this cell FIRST

import os

IN_COLAB = os.environ.get("COLAB_RELEASE_TAG") is not None

if IN_COLAB:
    # --- Google Colab setup ---
    get_ipython().system('pip install -q --upgrade yt-dlp')
    get_ipython().system('apt-get install -y ffmpeg')
    get_ipython().system('apt-get remove -y nodejs')
    get_ipython().system('apt-get purge -y nodejs')
    get_ipython().system('rm -rf /etc/apt/sources.list.d/nodesource.list')
    get_ipython().system('curl -sL https://deb.nodesource.com/setup_lts.x | bash -')
    get_ipython().system('apt-get install -y nodejs')
    get_ipython().system('node -v')
    get_ipython().system('which node')
else:
    # --- Local setup (Windows/Mac/Linux) ---
    get_ipython().system('pip install -q --upgrade yt-dlp')
    print("Local mode detected.")
    print("Make sure ffmpeg is installed and available on PATH.")
    print("Windows: https://ffmpeg.org/download.html")
    print("Then restart the kernel and continue from Cell 1.")

Local mode detected.
Make sure ffmpeg is installed and available on PATH.
Windows: https://ffmpeg.org/download.html
Then restart the kernel and continue from Cell 1.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Run this cell SECOND — after installs finish

import os
import shutil
from pathlib import Path

import yt_dlp

IN_COLAB = os.environ.get("COLAB_RELEASE_TAG") is not None

if IN_COLAB:
    from google.colab import files
    output_dir = "/content/songs"
else:
    files = None
    output_dir = os.path.join(os.getcwd(), "songs")

os.makedirs(output_dir, exist_ok=True)


def resolve_ffmpeg_location():
    """Find a folder containing both ffmpeg and ffprobe."""
    env_loc = os.environ.get("FFMPEG_LOCATION") or os.environ.get("FFMPEG_PATH")
    candidates = []
    if env_loc:
        candidates.append(Path(env_loc))
    candidates.extend([
        Path(r"C:\ffmpeg\bin"),
        Path(r"C:\Tools\ffmpeg\bin"),
        Path("/usr/bin"),
        Path("/usr/local/bin"),
    ])

    for p in candidates:
        if p.is_dir():
            d = p
        elif p.is_file():
            d = p.parent
        else:
            continue
        if (d / "ffmpeg.exe").exists() and (d / "ffprobe.exe").exists():
            return str(d)
        if (d / "ffmpeg").exists() and (d / "ffprobe").exists():
            return str(d)

    ffmpeg = shutil.which("ffmpeg")
    ffprobe = shutil.which("ffprobe")
    if ffmpeg and ffprobe:
        return str(Path(ffmpeg).parent)
    return None


ffmpeg_location = resolve_ffmpeg_location()

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Output folder: {output_dir}")
if ffmpeg_location:
    print(f"FFmpeg found: {ffmpeg_location}")
else:
    print("WARNING: FFmpeg not found — MP3 conversion will fail.")
    print("Install from https://ffmpeg.org/download.html or set FFMPEG_LOCATION.")

Environment: Local
Output folder: c:\Users\Kamlesh\Downloads\songs


In [7]:
# Bootstrap — run Cell 1 first, but this fills in missing variables if you skipped it
import os
import shutil
from pathlib import Path

import yt_dlp

if 'IN_COLAB' not in globals():
    IN_COLAB = os.environ.get("COLAB_RELEASE_TAG") is not None

if 'output_dir' not in globals():
    output_dir = "/content/songs" if IN_COLAB else os.path.join(os.getcwd(), "songs")
    os.makedirs(output_dir, exist_ok=True)

if 'ffmpeg_location' not in globals():
    def resolve_ffmpeg_location():
        env_loc = os.environ.get("FFMPEG_LOCATION") or os.environ.get("FFMPEG_PATH")
        candidates = []
        if env_loc:
            candidates.append(Path(env_loc))
        candidates.extend([
            Path(r"C:\ffmpeg\bin"),
            Path(r"C:\Tools\ffmpeg\bin"),
            Path("/usr/bin"),
            Path("/usr/local/bin"),
        ])
        for p in candidates:
            if p.is_dir():
                d = p
            elif p.is_file():
                d = p.parent
            else:
                continue
            if (d / "ffmpeg.exe").exists() and (d / "ffprobe.exe").exists():
                return str(d)
            if (d / "ffmpeg").exists() and (d / "ffprobe").exists():
                return str(d)
        ffmpeg = shutil.which("ffmpeg")
        ffprobe = shutil.which("ffprobe")
        if ffmpeg and ffprobe:
            return str(Path(ffmpeg).parent)
        return None

    ffmpeg_location = resolve_ffmpeg_location()

print(f"Output folder: {output_dir}")
print(f"FFmpeg: {ffmpeg_location or 'NOT FOUND'}")

# List of song search terms
songs = [
"PHIR SE| Dhurandhar The Revenge ",
"JAIYE SAJANA : Dhurandhar The Revenge",
"Ishq Jalakar - Karvaan | Dhurandhar ",
"JAAN SE GUZARTE HAIN | Dhurandhar The Revenge ",
"Comedy, Music And Unlimited Laughter! The Great Indian Kapil Show",
"Karma Video Song (Hindi) - Kantara Chapter 1 ",
"Rebel Lyrical Song (Hindi) - Kantara Chapter 1 | Rishab Shetty, Diljit Dosanjh ",
"'Birju' Video Song | Mika Singh, Udit Narayan",
"Meri Jindagi Se Jaane Ka Kya Loge Tum ",
"Kasto Mazza | Sonu Nigam & Shreya Ghoshal",
"TERA CHEHRA(Title Track): Adnan Sami ",
"Ilahi Full song Yeh Jaawani Hai Deewani",
"Highway: 'Maahi Ve' Full Song By A. R. Rehman",
"ROCKSTAR: Phir Se Ud Chala (Full Song) |A. R. Rahman",
"yuhi chala chal rahi HQ",
"Sajanaji Vari Vari Full Song",
"Makhna - Drive ",
"Ikk Kudi -| Udta Punjab | ",
"Dil Jhoom | Gadar 2 | Arijit Singh |",
"Qaafirana | Kedarnath |",
"Monta Re | Ananya Chakraborty with #soundsofisha | Amit Trivelli | Live at MahaShivratri 2023",
"Chaap Tilak -(Official Video) Sajid Wajid | Danish Sabri | Salman Ali | Shabab Sabri | Taaleem Music",
"Shakira - Waka Waka",
"Ed Sheeran - Sapphire",
"Dekh Tuni Bayko | Superhit Ahirani Song |",
"Zat Pat Pata Pat- Danny Pandit"

]

# yt-dlp config
ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': os.path.join(output_dir, '%(title)s.%(ext)s'),
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
    'noplaylist': True,
    'concurrent_fragments': 10,
    'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
    'referer': 'https://www.youtube.com/',
    'add_header': ['Accept-Language: en-US,en;q=0.5', 'Accept-Encoding: gzip, deflate, br'],
    'js_runtimes': {'node': {}},
}

if ffmpeg_location:
    ydl_opts['ffmpeg_location'] = ffmpeg_location
else:
    raise RuntimeError(
        "FFmpeg not found. Install it or set FFMPEG_LOCATION to the folder "
        "containing ffmpeg.exe and ffprobe.exe (e.g. C:\\ffmpeg\\bin)"
    )

# Download MP3s
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    for song in songs:
        print(f"\nDownloading: {song}")
        try:
            ydl.download([f"ytsearch1:{song}"])
        except Exception as e:
            print(f"Error downloading {song}: {e}")

Output folder: c:\Users\Kamlesh\Downloads\songs
FFmpeg: C:\ffmpeg\bin

Downloading: PHIR SE| Dhurandhar The Revenge 
[youtube:search] Extracting URL: ytsearch1:PHIR SE| Dhurandhar The Revenge 
[download] Downloading playlist: PHIR SE| Dhurandhar The Revenge 
[youtube:search] query "PHIR SE| Dhurandhar The Revenge ": Downloading web client config


[youtube:search] query "PHIR SE| Dhurandhar The Revenge " page 1: Downloading API JSON
[youtube:search] Playlist PHIR SE| Dhurandhar The Revenge : Downloading 1 items of 1
[download] Downloading item 1 of 1
[youtube] Extracting URL: https://www.youtube.com/watch?v=r8O3URprq1M
[youtube] r8O3URprq1M: Downloading webpage
[youtube] r8O3URprq1M: Downloading android vr player API JSON
[info] r8O3URprq1M: Downloading 1 format(s): 251
[download] c:\Users\Kamlesh\Downloads\songs\PHIR SE (Full Video) ｜ Dhurandhar The Revenge ｜ Ranveer Singh ｜ Shashwat Sachdev,Arijit S,Irshad K.webm has already been downloaded
[download] 100% of    3.80MiB
[ExtractAudio] Destination: c:\Users\Kamlesh\Downloads\songs\PHIR SE (Full Video) ｜ Dhurandhar The Revenge ｜ Ranveer Singh ｜ Shashwat Sachdev,Arijit S,Irshad K.mp3
Deleting original file c:\Users\Kamlesh\Downloads\songs\PHIR SE (Full Video) ｜ Dhurandhar The Revenge ｜ Ranveer Singh ｜ Shashwat Sachdev,Arijit S,Irshad K.webm (pass -k to keep)
[download] Finished d

In [ ]:
# ZIP files

output_zip = "collab_songs"

shutil.make_archive(output_zip, 'zip', output_dir)

print("Folder zipped successfully!")
print(f"Zip file: {os.path.abspath(output_zip + '.zip')}")

Folder zipped successfully!


In [ ]:
# Download files to your device (Colab only)

if IN_COLAB:
    for file in os.listdir(output_dir):
        if file.endswith(".mp3"):
            files.download(os.path.join(output_dir, file))
else:
    print(f"Local mode — MP3 files are already on your PC:")
    for file in os.listdir(output_dir):
        if file.endswith(".mp3"):
            print(f"  {os.path.join(output_dir, file)}")

In [ ]:
# Optional: save to Google Drive (Colab only)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Skip this cell locally — Google Drive mount works only in Colab.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import shutil

# # Define the source (the zipped file) and destination (a folder in your Drive)
# source_zip_path = '/content/collab_songs.zip'
# destination_drive_path = '/content/drive/MyDrive/songs_from_gcolab'

# # Create the destination directory in Drive if it doesn't exist
# os.makedirs(destination_drive_path, exist_ok=True)

# # Move the zipped file to Google Drive
# shutil.move(source_zip_path, destination_drive_path)

# print(f"'{source_zip_path}' moved to '{destination_drive_path}'")

In [ ]:
# pip install spotipy


In [ ]:
# import spotipy
# from spotipy.oauth2 import SpotifyClientCredentials

# CLIENT_ID = "YOUR_CLIENT_ID"
# CLIENT_SECRET = "YOUR_CLIENT_SECRET"

# auth_manager = SpotifyClientCredentials(
#     client_id="04894ce7debd4651bd4e030395326225",
#     client_secret="030ecf9eeb4645abaf1bb3de50713841"
# )

# sp = spotipy.Spotify(auth_manager=auth_manager)

# playlist_url = "https://open.spotify.com/playlist/049FyxDVTkrmqqU5pzjT8C?si=h_YpsDCwRleldKvhcBMf_g&pi=PdKmh4zHScuHF"

# results = sp.playlist_tracks(playlist_url)

# songs = []

# while results:
#     for item in results['items']:
#         track = item['track']
#         if track:
#             song_name = track['name']
#             artist = track['artists'][0]['name']
#             songs.append(f"{song_name} - {artist}")

#     # pagination (important for playlists >100 songs)
#     if results['next']:
#         results = sp.next(results)
#     else:
#         results = None


# for i, song in enumerate(songs, 1):
#     print(f"{i}. {song}")

# print(f"\nTotal Songs: {len(songs)}")
